# Private Score 재현 노트북 (외부 모델 없음)

- 입력: `train.csv`, `test_private.csv`만 사용 (외부 데이터 없음)
- 평가 데이터는 **학습에 사용하지 않음**: 학습 행은 train 1~921 (922~924행은 lead feature가 test 행을 참조하므로 제외)
- Target_1: `conv1d` (lead feature 입력 신경망, 시드 5개 평균) + XGBoost(fi_al) 순위 평균 블렌드
- 검증(walk-forward): 블렌드 wf5 0.804 / wf10 0.806, fold 최저 0.695 / 0.649 (`results/neural_lead_T1*.json`, `results/lead_val_xgb.json`)
- Target_2: XGBoost (level+change + 전체 lead k=1..3), 시드 5개 평균
- 검증 근거: walk-forward 5 fold (`record.md`, `report.html`)
- 재현: 시드 고정. MPS는 완전 결정적이지 않아 소수점 셋째 자리 수준의 차이가 있을 수 있음. 완전 재현이 필요하면 `DEVICE="cpu"`


In [1]:
import xgboost  # macOS libomp 충돌 방지: 반드시 torch보다 먼저 import (torch 먼저 올리면 커널 크래시)
from xgboost import XGBClassifier
import copy, json, random, sys, time
import numpy as np, pandas as pd, torch, torch.nn.functional as F, sklearn
from src.data import ROOT, TARGETS, load
from src.lead import MAX_LEAD, lead_frame, streams
from src.neural_lead import MODELS
from src.train import windows

SEEDS, W, DEVICE = 5, 10, "mps" if torch.backends.mps.is_available() else "cpu"
T1_MODEL, KD, KD_ALPHA, BLEND = "conv1d", False, 0.3, True
CFG = dict(lr=1e-3, wd=2e-2, bs=64, max_epochs=120, patience=15, noise=0.15, feat_drop=0.2, step_drop=0.15, drop=0.3, d=64, L=2)
def seed_all(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
print("python", sys.version.split()[0], "| torch", torch.__version__, "| xgboost", xgboost.__version__, "| sklearn", sklearn.__version__, "| device", DEVICE)

df, is_train = load()
y = df.loc[is_train, TARGETS].to_numpy().astype(np.float32)
n = int(is_train.sum())
train_idx, test_idx = np.where(is_train)[0], np.where(~is_train)[0]
fit_idx = train_idx[:-MAX_LEAD]          # 922~924 제외
fit_mask = np.zeros(len(df), bool); fit_mask[fit_idx] = True
print("train rows used:", len(fit_idx), "| test rows:", len(test_idx))

python 3.12.13 | torch 2.14.1 | xgboost 3.4.1 | sklearn 1.9.1 | device mps
train rows used: 921 | test rows: 200


## Target_2: XGBoost

In [2]:
X2 = lead_frame(df, is_train, "all_k123+lc", fit_mask).to_numpy(np.float32)
XGB = dict(n_estimators=400, learning_rate=0.03, max_depth=3, subsample=0.8, colsample_bytree=0.5, min_child_weight=5, reg_lambda=1.0, n_jobs=8)
pred_2 = np.mean([XGBClassifier(**XGB, random_state=s).fit(X2[fit_idx], y[:len(fit_idx), 1]).predict_proba(X2[test_idx])[:, 1] for s in range(5)], 0)
print("T2 n_feat", X2.shape[1], "| pred mean", pred_2.mean().round(3))

T2 n_feat 1976 | pred mean 0.481


## Target_1: lead feature 신경망

In [3]:
S = streams(df, is_train, fit_mask)
X, dims = {}, {}
for g, A in S.items():
    A = A.fillna(0).to_numpy(np.float32); mu, sd = A[fit_mask].mean(0), A[fit_mask].std(0) + 1e-6
    X[g] = torch.as_tensor(windows(np.clip((A - mu) / sd, -5, 5), W), device=DEVICE); dims[g] = A.shape[1]
take = lambda idx: {g: x[idx] for g, x in X.items()}
def augment(xb):
    out = {}
    for g, x in xb.items():
        x = x + CFG["noise"] * torch.randn_like(x)
        x = x * (torch.rand(x.shape[0], 1, x.shape[2], device=DEVICE) > CFG["feat_drop"])
        m = (torch.rand(x.shape[0], x.shape[1], 1, device=DEVICE) > CFG["step_drop"]).float(); m[:, -1] = 1
        out[g] = x * m
    return out
@torch.no_grad()
def predict(model, idx):
    model.eval(); return torch.sigmoid(model(take(torch.as_tensor(idx, device=DEVICE)))).cpu().numpy()
yt = torch.as_tensor(y[:, 0], device=DEVICE)
tt = torch.as_tensor(np.nan_to_num(teacher, nan=-1.0), device=DEVICE) if KD else None
tr_in, es = fit_idx[:-(60 + MAX_LEAD)], fit_idx[-60:]   # early stopping용 inner tail
def fit(seed):
    seed_all(seed); rng = np.random.default_rng(seed)
    model = MODELS[T1_MODEL](dims, d=CFG["d"], L=CFG["L"], drop=CFG["drop"], W=W).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=CFG["lr"], weight_decay=CFG["wd"])
    best, state, wait = -1, None, 0
    for ep in range(CFG["max_epochs"]):
        model.train()
        for s in range(0, len(tr_in), CFG["bs"]):
            bi = torch.as_tensor(rng.permutation(tr_in)[s:s + CFG["bs"]], device=DEVICE)
            logit = model(augment(take(bi))); loss = F.binary_cross_entropy_with_logits(logit, yt[bi])
            if tt is not None and (has := tt[bi] >= 0).any():
                loss = KD_ALPHA * loss + (1 - KD_ALPHA) * F.binary_cross_entropy_with_logits(logit[has], tt[bi][has])
            opt.zero_grad(); loss.backward(); opt.step()
        from sklearn.metrics import roc_auc_score
        auc = roc_auc_score(y[es, 0], predict(model, es))
        if auc > best: best, state, wait = auc, copy.deepcopy(model.state_dict()), 0
        elif (wait := wait + 1) >= CFG["patience"]: break
    model.load_state_dict(state); return model
t0 = time.time()
pred_1_nn = np.mean([predict(fit(s), test_idx) for s in range(SEEDS)], 0)
print(f"T1 {T1_MODEL} dims={dims} | pred mean {pred_1_nn.mean():.3f} | {time.time() - t0:.0f}s")

T1 conv1d dims={'Q': 110, 'KE': 102, 'SH': 400, 'WE': 32} | pred mean 0.534 | 7s


## Target_1: XGBoost(fi_al) + 순위 평균 블렌드

In [4]:
from scipy.stats import rankdata
X1 = lead_frame(df, is_train, "fi_al", fit_mask).fillna(0).to_numpy(np.float32)
pred_1_xgb = np.mean([XGBClassifier(**XGB, random_state=s).fit(X1[fit_idx], y[:len(fit_idx), 0]).predict_proba(X1[test_idx])[:, 1] for s in range(5)], 0)
# 순위 평균을 [0,1]로 정규화 (AUC는 순위만 사용)
pred_1 = ((rankdata(pred_1_nn) + rankdata(pred_1_xgb)) / 2 - 1) / (len(test_idx) - 1) if BLEND else pred_1_nn
print("T1 xgb n_feat", X1.shape[1], "| corr(nn, xgb) = %.3f" % np.corrcoef(pred_1_nn, pred_1_xgb)[0, 1], "| blend:", BLEND)

T1 xgb n_feat 92 | corr(nn, xgb) = 0.894 | blend: True


## 제출 파일

In [5]:
sub = pd.DataFrame({"Time_step": df.loc[test_idx, "Time_step"].to_numpy(), "pred_1": pred_1, "pred_2": pred_2})
out = ROOT / "submissions"; out.mkdir(exist_ok=True)
for name, m, ref in [("public_3반_안중범_nokd.csv", 100, "sample_submission_public.csv"), ("private_3반_안중범_nokd.csv", 200, "sample_submission_private.csv")]:
    part = sub.iloc[:m]
    assert len(part) == m and (part["Time_step"].to_numpy() == pd.read_csv(ROOT / ref)["Time_step"].to_numpy()).all()
    assert part[["pred_1", "pred_2"]].apply(lambda c: c.between(0, 1).all()).all() and part.notna().all().all()
    part.to_csv(out / name, index=False); print("saved", out / name)

saved /Users/anjungbeom/Desktop/dataa/submissions/public_3반_안중범_nokd.csv
saved /Users/anjungbeom/Desktop/dataa/submissions/private_3반_안중범_nokd.csv
